In [ ]:
from re import T
import torch
import math


class LinearLayer:
    def __init__(self,in_features,out_features,bias=False):
        kaim = math.sqrt(2/in_features)
        self.weights = torch.randn(out_features,in_features)*kaim
        self.has_bias= bias
        if bias:
            self.bias = torch.zeros(out_features)
        else:
            self.bias = None

    def forward(self,x):
        self.x  = x
        out = self.x @ self.weights.T
        if self.has_bias:
            out = out+self.bias
        
        return out


    def backward(self,grad_out):
        x_shape = self.x.shape
        x_flat = self.x.flatten(0,-2)
        grad_flat = grad_out.flatten(0,-2)

        grad_inputs = grad_flat @ self.weights
        self.weights.grad = grad_flat.T @ x_flat
        if self.has_bias:
            self.bias.grad = grad_flat.sum(dim=0)

        grad_inputs = grad_inputs.reshape(x_shape)
        return grad_inputs


class LayerNorm:

    def __init__(self,num_dims,eps):
        self.eps = eps
        self.gamma = nn.Parameter(torch.ones(num_dims))
        self.beta = nn.Parameter(torch.zeros(num_dims))


    def forward(self,x):
        self.avg = torch.mean(x,dim=-1,keepdim=True)
        self.var = torch.mean( (x - self.avg)**2,dim=-1,keepdim=True )

        self.x_norm = (x - self.avg)/torch.sqrt(self.var+self.eps)
        out = self.x_norm * self.gamma + self.beta
        return out

    
    def backward(self, grad_out):
        self.grad_beta = torch.sum(grad_out, dim=(0, 1))
        self.grad_gamma = torch.sum(grad_out * self.x_norm, dim=(0, 1))
    
        grad_x = grad_out * self.gamma

        return (1.0 / torch.sqrt(self.var + self.eps)) * (
            grad_x
            - torch.mean(grad_x, dim=-1, keepdim=True)
            - self.x_norm * torch.mean(grad_x * self.x_norm, dim=-1, keepdim=True)
        )
        
        

class Relu:
    def forward(self,x):
        self.x = x
        return torch.clamp(x,min=0)

    def backward(self,grad_out):
        return grad_out * (self.x > 0)



class Dropout:
    
    def __init__(self,p=0.1,training=True):
        self.p = p
        self.training = training
        self.mask = None
        

    def forward(self,x):
        if self.training:
            self.mask = ((torch.rand_like(x) > self.p).float())/(1.0-self.p)
            return x * self.mask
        else:
            self.mask = None
            return x   

    def backward(self,grad_out):
        if self.training:
            return grad_out * self.p
        else:
            return grad_out


        
class MyMlp:
    def __init__(self, in_features, hidden_features, out_features):
        self.linear1 = LinearLayer(in_features, hidden_features)
        self.relu = Relu()
        self.linear2 = LinearLayer(hidden_features, out_features)
        self.dropout = Dropout(p=0.1, training=True)

    def forward(self, x):
        x = self.linear1.forward(x)
        x = self.relu.forward(x)
        x = self.linear2.forward(x)
        x = self.dropout.forward(x)
        return x

    def backward(self, grad_out):
        grad_drop = self.dropout.backward(grad_out)
        grad_linear2 = self.linear2.backward(grad_drop)
        grad_relu = self.relu.backward(grad_linear2)
        grad_linear1 = self.linear1.backward(grad_relu)
        return grad_linear1


class softmax:

    def forward(self,scores):
        max_score = torch.max(scores,dim=-1,keepdim=True).values
        scores_exp = torch.exp(scores-max_score)
        self.scores_sum = scores_exp.sum(dim=-1,keepdim=True)
        self.out = scores_exp/self.scores_sum
        return self.out

    
    def backward(self,grad_attn):

        sum_term = (grad_attn *self.out ).sum(dim=-1, keepdim=True)
        grad_scores = self.out * (grad_attn - sum_term)

        return grad_scores
         

class CausalSelfAttention:
    def __init__(self,num_dims,num_heads):
        self.num_dims = num_dims
        self.num_heads = num_heads
        self.head_dims = num_dims // num_heads

        self.w_q = LinearLayer(num_dims,num_dims,bias=False)
        self.w_k = LinearLayer(num_dims,num_dims,bias=False)
        self.w_v = LinearLayer(num_dims,num_dims,bias=False)

        self.proj_out = LinearLayer(num_dims,num_dims,bias=True)
        self.softmax = softmax()

        self.attn_drop = Dropout(p=0.1,training=True)
        self.resid_drop = Dropout(p=0.1,training=True)

    def forward(self,x):
        B,T,D = x.shape

        Q = self.w_q.forward(x)
        K = self.w_k.forward(x)
        V = self.w_v.forward(x)

        Q = Q.view(B,T,self.num_heads,self.head_dims).transpose(1,2)
        K = K.view(B,T,self.num_heads,self.head_dims).transpose(1,2)
        V = V.view(B,T,self.num_heads,self.head_dims).transpose(1,2)

        self.Q = Q
        self.K = K
        self.V = V
        
        scores = Q @ K.transpose(-2,-1) / math.sqrt(self.head_dims)
        masks = torch.triu(torch.ones(T,T,dtype=torch.bool,device=scores.device),diagonal=1)

        scores = scores.masked_fill(masks,float("-inf"))

        self.attn_scores = self.softmax.forward(scores)
        self.attn_scores = self.attn_drop.forward(self.attn_scores)
        self.out = self.attn_scores @ self.V
        self.out = self.out.transpose(1,2).contiguous().view(B,T,D)
        self.out = self.proj_out(self.out)
        self.out = self.resid_drop.forward(self.out)
        return self.out

    def backward(self,grad_out):
        B,T,D = grad_out.shape
        grad_resid = self.resid_drop.backward(grad_out)
        grad_proj = self.proj_out.backward(grad_resid)
        grad_proj = grad_proj.view(B,T,self.num_heads,self.head_dims).transpose(1,2)
        
        grad_attn = grad_proj @ self.V.transpose(-2,-1)
        
        grad_V = self.attn_scores.transpose(-2,-1) @ grad_proj
        
        grad_probs = self.attn_drop.backward(grad_attn)
        grad_scores = self.softmax.backward(grad_probs)   

        scale = 1.0 / math.sqrt(self.head_dims)
        grad_scores = grad_scores * scale
        
        grad_q = grad_scores @ self.K
        grad_k = grad_scores.transpose(-2,-1) @ self.Q
        

        grad_Q = grad_Q.transpose(1, 2).contiguous().view(B, T, D)
        grad_K = grad_K.transpose(1, 2).contiguous().view(B, T, D)
        grad_V = grad_V.transpose(1, 2).contiguous().view(B, T, D)

        grad_x_q = self.w_q.backward(grad_Q)
        grad_x_k = self.w_k.backward(grad_K)
        grad_x_v = self.w_v.backward(grad_V)

        grad_x = grad_x_q+grad_x_k+grad_x_v

        return grad_x

    def parameters(self):
        return [self.w_q.weights, self.w_k.weights, self.w_v.weights, self.proj_out.weights, self.proj_out.bias]
        

        
    

        

we use a special method called dropouts
we often encourage to use the dropouts to randomly fuse some percent of the neurons,because there are some neurons which would be lazy enough to work and try to copy the other neuron's work
for ex:
y1->is a features of the neuron 1
y2->is a feature of the neuron 2
y3 -> y1+y2 and this isn't actually creating a new features but it is just using other neurons work and we can fuse this neuron
but in the model we can't exactly know which neuron is being redundant so for that we fuse them randomly to learn the features by themselves

where do we actually use this dropouts ??
1)attention (after the softmax)
okay wait?why do we need to use after softmax,because there is a reason for that
if you use before the softmax,then they will become 0 and then e^0 is 1 right at the softmax they will revive using the reincarnation justu 😂️ .... ,so we don't prefer them to come alive
and after the softmax,we have done the masking for the next tokens and it can be sum up to the perfect 1 and this is the perfect place to use the dropout
because we are about to multiply with the V,so before the models learns to find the V,as they are the real content right,so for that we need to make sure the model uses the dims perfeclty so we use the dropout at that place
It prevents the model from obsessing over just one specific token in the context. By randomly zeroing out attention connections, it forces the query token to learn to attend to multiple different context tokens.

and the next position where we do actually fuses the tokens is
in the just before add the residual out to the original tokens

you might encounter these in the future,as we step down a few blocks,so just skip this part and you can come later too
we will soon understand why do we actually mix them up (spoilers: because we still need the originality of x and our predictions to make the model better at predicting the words)
and at the mlp stage,we do use dropout before adding to the x

At the very entrance of the model, right after adding token embeddings and positional embeddings:

$$x = \text{tok\_emb} + \text{pos\_emb}$$

$$x = \text{dropout}(x)$$


Now coming to the real layernorms work
the reason for the layernorms is to ensure that the activation don't either blew up or else vanish

gemini generated this so good,as you can consider this example:

The Microphone and Amplifier ProblemImagine a chain of 12 speakers and amplifiers in a room.
The first amplifier boosts the audio signal by just 1.2x.The next amplifier boosts it by 1.2x.By the 12th amplifier, the sound volume is $1.2^{12} \approx 8.9\text{x}$ louder.If one amp slips and boosts by 2x, $2^{12} = 4096\text{x}$. The sound turns into deafening screeches and distortion.

In a deep neural network, every layer multiplies matrices and adds residual vectors.
As the activations travel deeper through the network:Without control, numbers either blow up toward infinity (exploding activations) or shrink toward zero (vanishing activations).
When values blow up, mathematical operations like $\text{Softmax}$ choke: the largest number gets a probability of $1.0$, all others become $0.0$, and gradients drop dead to zero.

LayerNorm acts as an automatic sound engineer. Right before the signal enters an Attention or MLP block, LayerNorm grabs that specific token vector, centers its volume to 0, and normalizes its loudness to 1.

Where is it placed? (Pre-LN vs. Post-LN)
In early Transformers (original "Attention is All You Need"), normalization was applied after the residual add (Post-LN):

$$x = \text{LayerNorm}(x + \text{SubLayer}(x))$$

This caused training instability because backpropagating gradients had to fight through the normalization at every single residual step.Modern architectures like GPT-2, GPT-3, and LLaMA use Pre-LN:

$$x = x + \text{SubLayer}(\text{LayerNorm}(x))$$

In Pre-LN:
Clean Highway: The residual stream $x$ remains an untouched, clean highway where gradients can flow from the end of the model all the way to token embeddings without restriction.
Safe Input: Each sub-layer (Attention and MLP) receives an input that is cleanly scaled and normalized so it never receives exploding numbers.


in the layernorms the single tokens have the entire universe
it was like sending a single token and then finding the mean and var on that and we are gonna implemenet those calculations and send out that single token
so this happens for the all the tokens inside that

so we will work on the backward for that layernorms
okay i think we are ready for the backward right
didn't we ??
for the backward right we actually have the
grad_out right
and then the grad_out comes in
and then checking from the back means the one which is at the end is the
so we did retuned the out right
then,the main work is ,it will see the out

$$out = \text{self.x\_norm} \cdot \text{self.gamma} + \text{self.beta}$$

and now we need to find the derivative
wrt to the x_norm and then gamma and beta??
i mean beta and gamma are intially the 1's and then 0's so now the model slowly understand what to chane with them rih?

$$grad\_beta = grad\_out \cdot 1$$

so we sum it up
just like the bias right

$$\text{grad\_beta} = \text{torch.sum}(\text{grad\_out}, \text{dim}=(0,1))$$

for the
$$grad\_gamma = grad\_out \cdot x$$

Again, summing over dimensions $0$ and $1$:

$$\text{self.grad\_gamma} = \text{torch.sum}(\text{grad\_out} \cdot \text{self.x\_norm}, \text{dim}=(0, 1))$$


okay split this into two parts

$$\hat{x} = \frac{x - \mu}{\sqrt{\sigma^2 + \epsilon}}$$

and x-mean is the u
the leftover is v
so for now,we can find the derivative of the uv1+vu' right

the derivation will be done by gemini
as my skills in the mathematics arent that strong enough to make this for now

You are on the right conceptual path by breaking the term into a numerator $u = x - \mu$ and a denominator factor $v = (\sigma^2 + \epsilon)^{-1/2}$.

However, treating it as a simple single-variable product rule $(uv)' = u'v + uv'$ directly with respect to a single scalar $x$ will miss an important detail: **multivariable interaction**.

When you shift an element $x_i$, that change modifies the overall mean $\mu = \frac{1}{D}\sum_j x_j$ and the overall variance $\sigma^2 = \frac{1}{D}\sum_j (x_j - \mu)^2$. Because $\mu$ and $\sigma^2$ are functions of *all* elements in that row, every element $x_i$ interacts with every other element.


### Step 1: Upstream Gradient into $\hat{x}$

Let $d\hat{x}$ be the error flowing into the normalized value $\hat{x}$:

$$d\hat{x} = \text{grad\_out} \odot \gamma$$

Now trace the error backward through the computational graph from $\hat{x}$ back to $x$.


### Step 2: The Three Paths from $\hat{x}$ to $x$

$$\hat{x} = (x - \mu) \cdot (\sigma^2 + \epsilon)^{-1/2}$$

When computing $\frac{\partial \mathcal{L}}{\partial x}$, the chain rule splits across three paths:

1. **Path 1: Direct path via the numerator**

$$\left(\frac{\partial \mathcal{L}}{\partial x}\right)_{\text{direct}} = d\hat{x} \cdot \frac{1}{\sqrt{\sigma^2 + \epsilon}}$$


2. **Path 2: Through the mean $\mu$**

Since $\frac{\partial \mu}{\partial x_i} = \frac{1}{D}$, the total gradient flowing into the mean is the sum of gradients across all $D$ channels:

$$\frac{\partial \mathcal{L}}{\partial \mu} = \sum_{j=1}^D d\hat{x}_j \cdot \left(-\frac{1}{\sqrt{\sigma^2 + \epsilon}}\right) = -\frac{1}{\sqrt{\sigma^2 + \epsilon}} \sum_{j=1}^D d\hat{x}_j$$

Since each $x_i$ contributes $\frac{1}{D}$ to $\mu$:

$$\left(\frac{\partial \mathcal{L}}{\partial x_i}\right)_{\text{via } \mu} = -\frac{1}{\sqrt{\sigma^2 + \epsilon}} \cdot \frac{1}{D} \sum_{j=1}^D d\hat{x}_j = -\frac{1}{\sqrt{\sigma^2 + \epsilon}} \cdot \text{mean}(d\hat{x})$$


3. **Path 3: Through the variance $\sigma^2$ (Your $v'$ term)**

Differentiating $v = (\sigma^2 + \epsilon)^{-1/2}$ gives:

$$\frac{\partial v}{\partial \sigma^2} = -\frac{1}{2}(\sigma^2 + \epsilon)^{-3/2}$$

And $\frac{\partial \sigma^2}{\partial x_i} = \frac{2}{D}(x_i - \mu)$.
Tracing this through yields:

$$\left(\frac{\partial \mathcal{L}}{\partial x_i}\right)_{\text{via } \sigma^2} = -\frac{1}{\sqrt{\sigma^2 + \epsilon}} \cdot \hat{x}_i \cdot \frac{1}{D}\sum_{j=1}^D (d\hat{x}_j \cdot \hat{x}_j) = -\frac{1}{\sqrt{\sigma^2 + \epsilon}} \cdot \hat{x}_i \cdot \text{mean}(d\hat{x} \odot \hat{x})$$


### Step 3: Factoring Out $\frac{1}{\sqrt{\sigma^2 + \epsilon}}$

Adding all three paths together:

$$\frac{\partial \mathcal{L}}{\partial x_i} = \frac{1}{\sqrt{\sigma^2 + \epsilon}} \Big( d\hat{x}_i - \text{mean}(d\hat{x}) - \hat{x}_i \cdot \text{mean}(d\hat{x} \odot \hat{x}) \Big)$$

Factoring it this way eliminates the need to compute separate Jacobians or write Python loops. In PyTorch code, it evaluates in three clean lines:
